# 20 — Qwen3.8-27B zero-shot CTD sanity check (Biowulf OnDemand)

Biowulf/Open OnDemand version of the strong-model go/no-go experiment. No Colab-specific paths, no Google Drive mount, and no 4-bit quantization. Intended for an A100 80 GB interactive Jupyter session.

Repository root: `/data/LunaLab/yoshi/llm-tuning-playground`

Primary quantities: **Clean**, **Distractor-5** (evidence selection), and **Hard no-path** (evidence sufficiency). One DiseaseID split, one seed, and 50 examples per condition are enough for the first decision.


In [ ]:
# Environment and paths — run this first.
import os
from pathlib import Path

ROOT = Path('/data/LunaLab/yoshi/llm-tuning-playground')
assert ROOT.exists(), f'Repository root not found: {ROOT}'

# Keep large Hugging Face artifacts on the LunaLab filesystem so they are reused across sessions.
HF_HOME = ROOT / 'hf_cache'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'hub')
HF_HOME.mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT / 'ctd_data'
OUT_DIR = ROOT / 'results' / '20_biowulf'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('HF_HOME:', HF_HOME)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# Install only if the OnDemand kernel does not already have recent packages.
# If you maintain a dedicated environment, install these there once and comment this cell out.
%pip -q install -U transformers accelerate sentencepiece requests pandas safetensors


In [ ]:
import re, gzip, random, requests
import numpy as np
import pandas as pd
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_NAME = 'Qwen/Qwen3.8-27B'
SPLIT = 'DiseaseID'
SEED = 1
N_EVAL = 50
MAX_NEW_TOKENS = 48

assert torch.cuda.is_available(), 'No CUDA GPU is visible. Launch the OnDemand Jupyter session with an A100 GPU.'
props = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(props.total_memory / 1024**3, 1))
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('Model:', MODEL_NAME)


In [ ]:
# CTD acquisition / parsing. Existing files under ROOT/ctd_data are reused.
CHEM_NAME = 'CTD_chem_gene_ixns.tsv.gz'
GD_NAMES = ['CTD_curated_genes_diseases.tsv.gz', 'CTD_genes_diseases.tsv.gz']

def valid_gzip(path, min_bytes=10000):
    path = Path(path)
    if not path.exists() or path.stat().st_size < min_bytes:
        return False
    try:
        with open(path, 'rb') as f:
            if f.read(2) != b'\x1f\x8b':
                return False
        with gzip.open(path, 'rb') as f:
            f.read(128)
        return True
    except Exception:
        return False

def ensure_file(names):
    if isinstance(names, str):
        names = [names]
    for name in names:
        p = DATA_DIR / name
        if valid_gzip(p):
            print('Found:', p)
            return p
    for name in names:
        dest = DATA_DIR / name
        for url in [f'https://ctdbase.org/reports/{name}', f'https://ctdbase.org/downloads/{name}']:
            try:
                print('Downloading:', url)
                with requests.get(url, stream=True, timeout=(20, 300), headers={'User-Agent':'Mozilla/5.0'}) as r:
                    r.raise_for_status()
                    with open(dest, 'wb') as f:
                        for chunk in r.iter_content(1024 * 1024):
                            if chunk:
                                f.write(chunk)
                if valid_gzip(dest):
                    return dest
            except Exception as e:
                print('failed:', type(e).__name__, str(e)[:120])
            dest.unlink(missing_ok=True)
    raise FileNotFoundError(names)

def read_ctd(path, expected):
    header = None
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if not line.startswith('#'):
                break
            s = line.lstrip('#').strip()
            if '\t' in s:
                cols = [x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected):
                    header = cols
    if header is None:
        raise ValueError(f'Header not found: {path}')
    return pd.read_csv(path, sep='\t', comment='#', compression='gzip', dtype=str,
                       low_memory=False, header=None, names=header)

cg = read_ctd(ensure_file(CHEM_NAME), ['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd = read_ctd(ensure_file(GD_NAMES), ['GeneSymbol','GeneID','DiseaseName','DiseaseID'])

cg2 = cg[['ChemicalName','ChemicalID','GeneSymbol','GeneID']].dropna().drop_duplicates()
gd2 = gd[['GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().drop_duplicates()
paths = cg2.merge(gd2, on=['GeneSymbol','GeneID']).drop_duplicates()
paths = paths[(paths.ChemicalName.str.len() < 100) & (paths.DiseaseName.str.len() < 120)].reset_index(drop=True)
edge_pool = gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)

print('paths:', len(paths), 'edges:', len(edge_pool))


In [ ]:
# One entity-disjoint test split; only the 3 conditions needed for the go/no-go check.
rng_np = np.random.default_rng(SEED)
entities = paths[SPLIT].dropna().unique().copy()
rng_np.shuffle(entities)
cut = max(1, int(0.8 * len(entities)))
test_entities = set(entities[cut:])
test_pool = paths[paths[SPLIT].isin(test_entities)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
assert len(test_pool) >= N_EVAL
test = test_pool.sample(N_EVAL, random_state=1000 + SEED).reset_index(drop=True)

def render(row, edges):
    ev = '\n'.join(f'- {g} -> {d}' for g, d in edges)
    return (
        'Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. '
        'If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'
        f'Chemical: {row.ChemicalName}\n'
        f'Gene: {row.GeneSymbol}\n'
        f'Evidence:\n{ev}'
    )

def positive_edges(row, k, rng):
    out = [(str(row.GeneSymbol), str(row.DiseaseName))]
    if k > 0:
        pool = edge_pool[(edge_pool.GeneSymbol != row.GeneSymbol) &
                         (edge_pool.DiseaseName != row.DiseaseName)]
        sub = pool.sample(k, random_state=rng.randint(0, 2**31 - 1))
        out += [(str(g), str(d)) for g, d in sub.itertuples(index=False, name=None)]
    rng.shuffle(out)
    return out

def no_path_edges(row, k, rng):
    pool = edge_pool[(edge_pool.GeneSymbol != row.GeneSymbol) &
                     (edge_pool.DiseaseName != row.DiseaseName)]
    sub = pool.sample(k, random_state=rng.randint(0, 2**31 - 1))
    out = [(str(g), str(d)) for g, d in sub.itertuples(index=False, name=None)]
    rng.shuffle(out)
    return out

rng = random.Random(20000 + SEED)
sets = {'clean': [], 'distractor_5': [], 'hard_no_path': []}
for _, row in test.iterrows():
    sets['clean'].append({'prompt': render(row, positive_edges(row, 0, rng)),
                          'target': str(row.DiseaseName), 'no_path': False})
    sets['distractor_5'].append({'prompt': render(row, positive_edges(row, 5, rng)),
                                 'target': str(row.DiseaseName), 'no_path': False})
    sets['hard_no_path'].append({'prompt': render(row, no_path_edges(row, 5, rng)),
                                 'target': None, 'no_path': True})

print({k: len(v) for k, v in sets.items()})


In [ ]:
# Load the model in BF16. This is preferable to 4-bit for the strong-model sanity check.
# A100 80 GB should have enough memory for a 27B BF16 model plus short generation.
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

print('loaded')
print('allocated GiB:', round(torch.cuda.memory_allocated() / 1024**3, 2))
print('reserved GiB:', round(torch.cuda.memory_reserved() / 1024**3, 2))


In [ ]:
def generate_one(prompt):
    messages = [{'role':'user', 'content':[{'type':'text', 'text':prompt}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt'
    )
    dev = next(model.parameters()).device
    inputs = {k: v.to(dev) if hasattr(v, 'to') else v for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    n = inputs['input_ids'].shape[-1]
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

def norm(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def score(item, pred):
    p = norm(pred)
    if item['no_path']:
        return int('no supported path' in p)
    return int(norm(item['target']) in p and 'no supported path' not in p)

RESULT_CSV = OUT_DIR / '20_qwen38_27b_results.csv'
SUMMARY_CSV = OUT_DIR / '20_qwen38_27b_summary.csv'

rows = []
for condition, items in sets.items():
    print('\n' + '=' * 70)
    print(condition)
    print('=' * 70)
    for i, item in enumerate(items):
        pred = generate_one(item['prompt'])
        ok = score(item, pred)
        rows.append({
            'model': MODEL_NAME, 'split': SPLIT, 'seed': SEED,
            'condition': condition, 'i': i, 'target': item['target'],
            'prediction': pred, 'correct': ok
        })
        if i < 3:
            print(i, ok, pred[:180])
        if (i + 1) % 10 == 0:
            pd.DataFrame(rows).to_csv(RESULT_CSV, index=False)
            print(f'checkpoint {i + 1}/{len(items)}')
    acc = np.mean([r['correct'] for r in rows if r['condition'] == condition])
    print('accuracy:', round(float(acc), 3))

df = pd.DataFrame(rows)
df.to_csv(RESULT_CSV, index=False)
summary = df.groupby('condition', as_index=False).correct.mean().rename(columns={'correct':'accuracy'})
summary.to_csv(SUMMARY_CSV, index=False)
display(summary)
print('Saved:', RESULT_CSV)


In [ ]:
# Descriptive go/no-go rule — not a statistical test.
m = dict(zip(summary.condition, summary.accuracy))
clean = m['clean']
d5 = m['distractor_5']
hard = m['hard_no_path']
gap = d5 - hard

print(f'Clean={clean:.3f} | D5={d5:.3f} | Hard NP={hard:.3f} | D5-Hard gap={gap:+.3f}')
if clean >= 0.95 and d5 >= 0.95 and hard >= 0.95:
    print('GO/NO-GO: near ceiling on the strong model -> broad reliability claim is weak for this simple task.')
elif d5 >= 0.90 and hard <= 0.80:
    print('GO/NO-GO: clear selection-sufficiency dissociation persists -> worth continuing.')
else:
    print('GO/NO-GO: intermediate result -> inspect predictions and repeat with a second model/family before deciding.')
